# Практическая работа 1. Часть 1

В этом ноутбуке исследуются базовые способы управления поведением языковой модели через OpenRouter. На серии учебных экспериментов проверяется влияние system prompt, длины контекста, параметров `temperature` и `max_tokens`, требований к формату ответа и их совместного использования в многошаговом диалоге.

Для каждого задания проводится не менее двух сценариев, сохраняются реальные ответы модели и формулируются краткие выводы по результатам экспериментов.

In [ ]:
# Установим необходимые зависимости
!pip install -q openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.1 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

# Получаем секреты, сохранённые в Google Colab.
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")

print("Секреты успешно загружены.")

Секреты успешно загружены.


In [ ]:
from openai import OpenAI

import pandas as pd
import json
import re

Инициализируем клиента и выполняем один тестовый запрос, чтобы удостовериться, что всё работает корректно.

In [ ]:
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# OPENROUTER_MODEL = "qwen/qwen3.8-27b:free" # для этой не работал провайдер во время выполнения моих запросов
# OPENROUTER_MODEL = "qwen/qwen3.6-plus:free" # эта уже недоступна для бесплатного использования, к сожалению
OPENROUTER_MODEL = "liquid/lfm-2.5-2.6b:free"

print("OpenRouter client initialized.")

OpenRouter client initialized.


In [ ]:
response = openrouter_client.chat.completions.create(
    model=OPENROUTER_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Ответь одним коротким предложением: что такое большая языковая модель?"
        }
    ],
    temperature=0,
)

print(response.choices[0].message.content)

Большая языковая модель — это компьютерная программа, обученная на огромном массиве текстов, чтобы понимать и генерировать человеческий язык.


### Задание 1. Системные промпты и роли

Цель эксперимента — проверить, как системный промпт влияет на стиль, лексику и структуру ответа языковой модели.

Для каждого пользовательского запроса используются четыре варианта:

1. без system prompt;
2. роль «преподаватель»;
3. роль «технический эксперт»;
4. роль «дружелюбный помощник».

Чтобы сравнение было корректным, во всех вариантах используется одна и та же модель и одинаковые параметры генерации. Меняется только системный промпт.

Проверяются два сценария:

* **Сценарий 1 — обучение ML:** объяснение понятия переобучения модели;
* **Сценарий 2 — техническое объяснение:** разница между синхронным и асинхронным выполнением кода в Python.

После выполнения сравниваются стиль ответа, используемая лексика, структура объяснения и пригодность ответа для обучения.


In [ ]:
system_prompts = {
    "Без system prompt": None,

    "Преподаватель": (
        "Ты преподаватель. Объясняй материал понятно и последовательно, "
        "используй простые формулировки, примеры и логичную структуру. "
        "Твоя цель — помочь человеку понять тему."
    ),

    "Технический эксперт": (
        "Ты технический эксперт. Давай точные и технически корректные ответы. "
        "Используй профессиональную терминологию и обращай внимание "
        "на важные технические детали."
    ),

    "Дружелюбный помощник": (
        "Ты дружелюбный помощник. Объясняй понятно, спокойно и неформально. "
        "Избегай излишне сложных формулировок и старайся сделать ответ "
        "лёгким для восприятия."
    ),
}

In [ ]:
# Два разных пользовательских запроса.
# Для каждого из них будут протестированы все четыре system prompt.

scenarios = {
    "Сценарий 1 — обучение ML": (
        "Объясни, что такое переобучение модели машинного обучения, "
        "почему оно возникает и как понять, что модель переобучилась."
    ),

    "Сценарий 2 — Python": (
        "Объясни разницу между синхронным и асинхронным выполнением кода "
        "в Python и приведи пример, когда стоит использовать каждый подход."
    ),
}

In [ ]:
def ask_model(user_prompt, system_prompt=None):
    """
    Отправляет запрос модели через OpenRouter
    """

    messages = []

    if system_prompt is not None:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=messages,
        temperature=0.2, # используем одинаковую temperature во всех экспериментах
    )

    return response.choices[0].message.content

In [ ]:
results = {}

for scenario_name, user_prompt in scenarios.items():

    print("=" * 100)
    print(scenario_name)
    print("=" * 100)
    print(f"\nПользовательский запрос:\n{user_prompt}\n")

    results[scenario_name] = {}

    for role_name, system_prompt in system_prompts.items():

        print("-" * 100)
        print(f"Вариант: {role_name}")
        print("-" * 100)

        answer = ask_model(
            user_prompt=user_prompt,
            system_prompt=system_prompt
        )

        # Сохраняем ответ для дальнейшего анализа.
        results[scenario_name][role_name] = answer

        print(answer)
        print()

Сценарий 1 — обучение ML

Пользовательский запрос:
Объясни, что такое переобучение модели машинного обучения, почему оно возникает и как понять, что модель переобучилась.

----------------------------------------------------------------------------------------------------
Вариант: Без system prompt
----------------------------------------------------------------------------------------------------
# Переобучение модели машинного обучения

## Что такое переобучение?

**Переобучение** (overfitting) — это ситуация, когда модель машинного обучения слишком хорошо «запоминает» данные обучающей выборки, а не учится общих закономерностям. В результате модель работает отлично на данных, с которыми она тренировалась, но плохо на новых, независимых данных (на тестовой выборке).

---

## Почему происходит переобучение?

### 1. **Слишком сложная модель**
- Модель имеет слишком много параметров по отношению к размеру обучающей выборки
- Например: очень глубокая нейросеть или линейная регрессия высок

In [ ]:
def save_results_to_markdown(results, filename="task1_results.md"):

    with open(filename, "w", encoding="utf-8") as f:

        f.write("# Задание 1. Результаты эксперимента\n\n")

        for scenario_name, scenario_results in results.items():
            f.write(f"## {scenario_name}\n\n")

            for role_name, answer in scenario_results.items():
                f.write(f"### {role_name}\n\n")
                f.write(answer.strip())
                f.write("\n\n---\n\n")

    print(f"Результаты сохранены в {filename}")

In [ ]:
save_results_to_markdown(results)

Результаты сохранены в task1_results.md


### Анализ результатов

В обоих сценариях system prompt заметно влиял прежде всего на **стиль, лексику и структуру ответа**, тогда как основное содержание оставалось похожим.

* **Без system prompt** модель давала нейтральный, подробный и хорошо структурированный ответ, но стиль выбирала самостоятельно.
* **«Преподаватель»** чаще использовал простые объяснения, аналогии и пошаговую подачу. Например, в сценарии с переобучением появилась аналогия с распознаванием кошек и более учебная структура.
* **«Технический эксперт»** давал более формальные ответы с большим количеством профессиональной терминологии и технических деталей. Это особенно заметно в сценарии про Python, где появились понятия `event loop`, `coroutines`, I/O-bound задачи и различие между конкурентностью и параллелизмом.
* **«Дружелюбный помощник»** использовал наиболее простой и разговорный стиль, сокращал технические детали и делал объяснение легче для восприятия.

Для обучения наиболее удачным оказался system prompt **«преподаватель»**, так как он лучше сочетал понятность, структуру и примеры.


### Вывод

Эксперимент показал, что system prompt эффективно управляет формой ответа модели.

Для **обучения новой теме** лучше подходит роль «преподаватель», для **технического разбора** — «технический эксперт», а роль «дружелюбный помощник» полезна, когда важна простота объяснения.

Таким образом, на качество ответа влияет не только сам пользовательский запрос, но и правильно заданная роль модели: она определяет уровень детализации, используемую лексику и способ подачи информации.


### Задание 2. Контекст и его длина

Цель эксперимента — проверить, как увеличение объёма контекста влияет на способность модели использовать информацию, находящуюся в его начале.

Для каждого сценария используются три размера контекста:

* короткий — около **50 токенов**;
* средний — около **200 токенов**;
* длинный — около **500 токенов**.

Проверяются два сценария:

1. запоминание кодового слова, указанного в начале контекста;
2. запоминание параметра объекта, также указанного в начале контекста.

Во всех случаях после важной информации добавляется нейтральный отвлекающий текст. Чем длиннее контекст, тем дальше нужный факт оказывается от пользовательского вопроса.

Дополнительно измеряется длина ответа модели, чтобы проверить, приводит ли увеличение входного контекста к увеличению объёма ответа.


Т.к. в задании требуется кол-во токенов, то добавим одну библиотеку. tiktoken использует не тот же токенизатор, что LFM2.5, поэтому число токенов будет приблизительным относительно самой модели. Но для эксперимента этого достаточно: мы получим одинаковый способ измерения для всех трёх вариантов.

In [ ]:
!pip install -q tiktoken

In [ ]:
import tiktoken

# Используем один и тот же токенизатор для оценки
# длины всех входных контекстов и ответов.
encoding = tiktoken.get_encoding("cl100k_base")


def count_tokens(text):
    """Приблизительно считает количество токенов в тексте."""
    return len(encoding.encode(text))


def build_context(initial_fact, filler, target_tokens):
    """
    Создаёт контекст примерно заданной длины.

    Важный факт всегда находится в самом начале,
    после него добавляется нейтральный текст.
    """

    text = initial_fact + "\n\n"

    while count_tokens(text) < target_tokens:
        text += filler + " "

    # Обрезаем текст до нужного количества токенов.
    tokens = encoding.encode(text)[:target_tokens]

    return encoding.decode(tokens)

In [ ]:
scenarios = {
    "Сценарий 1 — кодовое слово": {
        "fact": (
            "Важная информация: кодовое слово проекта — АМЕТИСТ. "
            "Запомни его, оно понадобится позже."
        ),
        "filler": (
            "Команда обсуждает этапы разработки проекта, проводит встречи, "
            "готовит документацию и распределяет задачи между участниками. "
            "Также рассматриваются сроки, ресурсы и возможные риски."
        ),
        "question": (
            "Какое кодовое слово проекта было указано в начале контекста? "
            "Ответь кратко."
        ),
        "expected": "АМЕТИСТ",
    },

    "Сценарий 2 — параметр сервера": {
        "fact": (
            "Важная информация: в тестовой системе сервер Orion "
            "использует порт 4812. Запомни этот параметр."
        ),
        "filler": (
            "В инфраструктуре используются различные сервисы и компоненты. "
            "Инженеры проверяют журналы событий, состояние контейнеров, "
            "сетевые соединения и доступность внутренних приложений."
        ),
        "question": (
            "Какой порт использует сервер Orion согласно информации "
            "в начале контекста? Ответь кратко."
        ),
        "expected": "4812",
    },
}

Здесь информация специально вымышленная. Нам важно не проверять знания модели, а заставить её использовать только предоставленный контекст.

In [ ]:
context_lengths = [50, 200, 500]

for length in context_lengths:
    context = build_context(
        initial_fact=scenarios["Сценарий 1 — кодовое слово"]["fact"],
        filler=scenarios["Сценарий 1 — кодовое слово"]["filler"],
        target_tokens=length,
    )

    print(f"Целевая длина: {length}")
    print(f"Полученная длина: {count_tokens(context)} токенов")
    print(context[:300])
    print("-" * 80)

Целевая длина: 50
Полученная длина: 50 токенов
Важная информация: кодовое слово проекта — АМЕТИСТ. Запомни его, оно понадобится позже.

Команда обсуждает эт
--------------------------------------------------------------------------------
Целевая длина: 200
Полученная длина: 200 токенов
Важная информация: кодовое слово проекта — АМЕТИСТ. Запомни его, оно понадобится позже.

Команда обсуждает этапы разработки проекта, проводит встречи, готовит документацию и распределяет задачи между участниками. Также рассматриваются сроки, ресурсы и возможные риски. Команда обсуждает этапы разрабо
--------------------------------------------------------------------------------
Целевая длина: 500
Полученная длина: 500 токенов
Важная информация: кодовое слово проекта — АМЕТИСТ. Запомни его, оно понадобится позже.

Команда обсуждает этапы разработки проекта, проводит встречи, готовит документацию и распределяет задачи между участниками. Также рассматриваются сроки, ресурсы и возможные риски. Команда обсуж

Здесь мы просто убеждаемся, что функция действительно создаёт тексты примерно на 50, 200 и 500 токенов.

In [ ]:
def ask_with_context(context, question):
    """
    Передаёт модели контекст и вопрос по нему.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "Ответь на вопрос, используя информацию из предоставленного "
                "контекста. Ответ должен быть кратким."
            ),
        },
        {
            "role": "user",
            "content": f"""
Контекст:
{context}

Вопрос:
{question}
""",
        },
    ]

    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=messages,
        temperature=0.2,
    )

    return response.choices[0].message.content

System prompt здесь одинаков для всех запусков. Поэтому изменяемый фактор — именно длина контекста.

In [ ]:
results = {}

for scenario_name, scenario in scenarios.items():

    print("=" * 100)
    print(scenario_name)
    print("=" * 100)

    results[scenario_name] = {}

    for target_length in context_lengths:

        context = build_context(
            initial_fact=scenario["fact"],
            filler=scenario["filler"],
            target_tokens=target_length,
        )

        answer = ask_with_context(
            context=context,
            question=scenario["question"],
        )

        answer_tokens = count_tokens(answer)

        # Простая автоматическая проверка,
        # содержится ли ожидаемый ответ в результате модели.
        remembered = scenario["expected"].lower() in answer.lower()

        results[scenario_name][target_length] = {
            "context_tokens": count_tokens(context),
            "answer": answer,
            "answer_tokens": answer_tokens,
            "remembered": remembered,
        }

        print(f"\nКонтекст: {target_length} токенов")
        print(f"Ответ: {answer}")
        print(f"Длина ответа: {answer_tokens} токенов")
        print(f"Нужная информация сохранена: {remembered}")

Сценарий 1 — кодовое слово

Контекст: 50 токенов
Ответ: АМЕТИСТ
Длина ответа: 7 токенов
Нужная информация сохранена: True

Контекст: 200 токенов
Ответ: АМЕТИСТ
Длина ответа: 7 токенов
Нужная информация сохранена: True

Контекст: 500 токенов
Ответ: АМЕТИСТ
Длина ответа: 7 токенов
Нужная информация сохранена: True
Сценарий 2 — параметр сервера

Контекст: 50 токенов
Ответ: Сервер Orion использует порт 4812.
Длина ответа: 12 токенов
Нужная информация сохранена: True

Контекст: 200 токенов
Ответ: Сервер Orion использует порт 4812.
Длина ответа: 12 токенов
Нужная информация сохранена: True

Контекст: 500 токенов
Ответ: Сервер Orion использует порт 4812.
Длина ответа: 12 токенов
Нужная информация сохранена: True


In [ ]:
rows = []

for scenario_name, scenario_results in results.items():
    for context_length, result in scenario_results.items():

        rows.append({
            "Сценарий": scenario_name,
            "Контекст, токенов": result["context_tokens"],
            "Ответ": result["answer"],
            "Ответ, токенов": result["answer_tokens"],
            "Информация сохранена": result["remembered"],
        })

results_df = pd.DataFrame(rows)

results_df

,Сценарий,"Контекст, токенов",Ответ,"Ответ, токенов",Информация сохранена
0,Сценарий 1 — кодовое слово,50,АМЕТИСТ,7,True
1,Сценарий 1 — кодовое слово,200,АМЕТИСТ,7,True
2,Сценарий 1 — кодовое слово,500,АМЕТИСТ,7,True
3,Сценарий 2 — параметр сервера,50,Сервер Orion использует порт 4812.,12,True
4,Сценарий 2 — параметр сервера,200,Сервер Orion использует порт 4812.,12,True
5,Сценарий 2 — параметр сервера,500,Сервер Orion использует порт 4812.,12,True


In [ ]:
def save_task2_results(results, filename="task2_results.md"):

    with open(filename, "w", encoding="utf-8") as f:

        f.write("# Задание 2. Результаты эксперимента\n\n")

        for scenario_name, scenario_results in results.items():

            f.write(f"## {scenario_name}\n\n")

            for context_length, result in scenario_results.items():

                f.write(
                    f"### Контекст: "
                    f"{result['context_tokens']} токенов\n\n"
                )

                f.write(f"**Ответ модели:**\n\n{result['answer']}\n\n")

                f.write(
                    f"**Длина ответа:** "
                    f"{result['answer_tokens']} токенов\n\n"
                )

                f.write(
                    f"**Информация сохранена:** "
                    f"{result['remembered']}\n\n"
                )

                f.write("---\n\n")

    print(f"Результаты сохранены в {filename}")

In [ ]:
save_task2_results(results)

Результаты сохранены в task2_results.md


### Анализ результатов

В обоих сценариях модель успешно сохранила нужную информацию при всех протестированных длинах контекста: 50, 200 и 500 токенов. Кодовое слово «АМЕТИСТ» было корректно восстановлено во всех трёх случаях, как и номер порта сервера Orion.
Следовательно, в пределах диапазона до 500 токенов признаков потери ранней информации обнаружено не было.

Длина ответа также не увеличивалась вместе с длиной контекста: в первом сценарии ответ во всех случаях занимал 7 токенов, во втором — 12 токенов. Это показывает, что увеличение входного контекста само по себе не приводит к более длинному ответу, особенно если явно задана инструкция отвечать кратко.


### Вывод

В проведённом эксперименте модель уверенно использовала информацию из начала контекста даже при его увеличении до 500 токенов. Момент, когда модель начинает терять контекст, в протестированном диапазоне обнаружить не удалось.

Также увеличение длины контекста с 50 до 500 токенов не повлияло на длину ответа. Таким образом, объём ответа в большей степени определяется инструкцией и характером вопроса, чем размером входного контекста.


### Задание 3. Эксперименты с гиперпараметрами

Цель эксперимента — проверить, как параметры генерации `temperature` и `max_tokens` влияют на ответы языковой модели.

Используются два сценария:

1. **Творческий сценарий** — генерация идеи AI-сервиса;
2. **Логический сценарий** — объяснение роли validation set в машинном обучении.

Эксперимент состоит из двух частей.

**Часть 1 — изменение `temperature`:**

* `0.0`
* `0.5`
* `1.0`

Для каждого значения выполняются два одинаковых запроса, чтобы оценить не только креативность и логичность, но и повторяемость результатов.

**Часть 2 — изменение `max_tokens`:**

* маленькое значение — 30;
* среднее — 100;
* большое — 250.

При исследовании одного параметра остальные настройки остаются неизменными. Это позволяет отдельно оценить влияние каждого гиперпараметра.


In [ ]:
scenarios = {
    "Сценарий 1 — творческий": (
        "Придумай идею небольшого AI-сервиса для студентов. "
        "Опиши проблему, которую он решает, принцип работы "
        "и одну интересную особенность сервиса."
    ),

    "Сценарий 2 — логический": (
        "Объясни, зачем в машинном обучении нужна отдельная validation set, "
        "если уже есть train set и test set. Приведи логичное обоснование."
    ),
}

Первый запрос намеренно открытый: модель может придумать много разных решений.

Второй, наоборот, имеет достаточно определённую смысловую основу, поэтому здесь можно посмотреть, не начинает ли высокая temperature ухудшать последовательность рассуждений.

**Примечание.** Используемая модель LFM2.5 выполняет внутреннее reasoning перед формированием финального ответа. Поэтому небольшие значения `max_tokens` могут полностью расходоваться на рассуждение или приводить к обрыву ответа.

В данном эксперименте используются следующие уровни:

* малый бюджет — `1000` токенов;
* средний — `2000` токенов;
* большой — `4000` токенов.

Эти значения выбраны экспериментально с учётом поведения используемой reasoning-модели. В части с `temperature` используется фиксированный лимит `4000`, чтобы длина генерации не была дополнительным изменяемым фактором.

In [ ]:
def generate_answer(prompt, temperature=0.0, max_tokens=4000):
    """
    Отправляет запрос модели с заданными параметрами генерации.

    У LFM2.5 max_tokens задаёт общий бюджет генерации,
    включая внутреннее reasoning.
    """

    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    message = response.choices[0].message

    return {
        "answer": message.content,
        "reasoning": getattr(message, "reasoning", None),
        "finish_reason": response.choices[0].finish_reason,
    }

In [ ]:
# Сначала проводим эксперимент с температурой
# Каждый запрос выполняем дважды

temperatures = [0.0, 0.5, 1.0]
n_repeats = 2

temperature_results = {}

for scenario_name, prompt in scenarios.items():

    print("=" * 100)
    print(scenario_name)
    print("=" * 100)

    temperature_results[scenario_name] = {}

    for temperature in temperatures:

        temperature_results[scenario_name][temperature] = []

        for run in range(1, n_repeats + 1):

            result = generate_answer(
                prompt=prompt,
                temperature=temperature,
                max_tokens=4000,  # фиксированный большой бюджет
            )

            temperature_results[scenario_name][temperature].append(result)

            print(f"\nTemperature = {temperature}, запуск {run}")
            print(f"finish_reason = {result['finish_reason']}")
            print("-" * 80)
            print(result["answer"])

Сценарий 1 — творческий

Temperature = 0.0, запуск 1
finish_reason = stop
--------------------------------------------------------------------------------
Вот идея для небольшего AI-сервиса, который может стать незаменимым помощником для студента.

### Название сервиса: «LexiLearn» (ЛексиКолл)

**Проблема:**
Студенты часто страдают от «информационного перегруза». У них есть куча разрозненных конспектов, но они не знают, как их структурировать для эффективного запоминания. Более того, традиционные методы (такие как простые карточки) требуют огромного времени на ручную подготовку и часто не учитывают индивидуальный темп усвоения материала. Студент может часами перечитать лекцию, не осознавая, что ему нужно повторить только конкретные узкие места.

**Принцип работы:**

1.  **Загрузка и анализ:** Пользователь загружает свои конспекты (PDF, сканы) или вводит текст лекции. AI-система использует модель NLP (обработки естественного языка), чтобы выделить ключевые термины, формулы и логические 

In [ ]:
# Теперь эксперимент с tokens

max_tokens_values = [1000, 2000, 4000]

max_tokens_results = {}

for scenario_name, prompt in scenarios.items():

    print("=" * 100)
    print(scenario_name)
    print("=" * 100)

    max_tokens_results[scenario_name] = {}

    for max_tokens in max_tokens_values:

        result = generate_answer(
            prompt=prompt,
            temperature=0.5,
            max_tokens=max_tokens,
        )

        max_tokens_results[scenario_name][max_tokens] = result

        print(f"\nmax_tokens = {max_tokens}")
        print(f"finish_reason = {result['finish_reason']}")
        print("-" * 80)
        print(result["answer"])

Сценарий 1 — творческий

max_tokens = 1000
finish_reason = length
--------------------------------------------------------------------------------
None

max_tokens = 2000
finish_reason = length
--------------------------------------------------------------------------------
Вот идея для небольшого AI-сервиса, ориентированного на оптимизацию учебного процесса.

### Название: **Synth** (Синтез)

**Проблема:**
Студенты часто сталкиваются

max_tokens = 4000
finish_reason = stop
--------------------------------------------------------------------------------
Вот идея для небольшого AI-сервиса, который может быть очень полезен студентам.

### Название: **«Lexi» (Словарь понимания)**

**Проблема:**
Студенты часто сталкиваются с проблемой «информационного перегруза». После лекции лектор говорит много, но студент не понимает, чем всё это отличается от других предметов, а также сложно структурировать хаотичные заметки так, чтобы увидеть логические связи между разными темами. Кроме того, стандарт

In [ ]:
def save_task3_results(
    temperature_results,
    max_tokens_results,
    filename="task3_results.md"
):
    """
    Сохраняет результаты задания 3 в Markdown-файл.

    В файл записываются:
    1. результаты эксперимента с temperature;
    2. результаты эксперимента с max_tokens;
    3. finish_reason для каждого запуска;
    4. фактическая длина финального ответа;
    5. информация о случаях, когда финальный ответ не был сформирован.
    """

    with open(filename, "w", encoding="utf-8") as f:

        f.write("# Задание 3. Результаты эксперимента\n\n")

        # ============================================================
        # Часть 1. Temperature
        # ============================================================

        f.write("## Часть 1. Temperature\n\n")

        for scenario_name, scenario_results in temperature_results.items():

            f.write(f"### {scenario_name}\n\n")

            for temperature, runs in scenario_results.items():

                f.write(f"#### temperature = {temperature}\n\n")

                for i, result in enumerate(runs, start=1):

                    answer = result["answer"]
                    finish_reason = result["finish_reason"]

                    if answer is not None:
                        actual_tokens = count_tokens(answer)
                    else:
                        actual_tokens = 0

                    f.write(f"**Запуск {i}**\n\n")

                    f.write(
                        f"- finish_reason: `{finish_reason}`\n"
                    )

                    f.write(
                        f"- длина финального ответа: "
                        f"≈ {actual_tokens} токенов\n\n"
                    )

                    f.write("**Ответ модели:**\n\n")

                    if answer is not None:
                        f.write(answer.strip())
                    else:
                        f.write(
                            "[Финальный ответ не сформирован: "
                            "лимит генерации был исчерпан до появления content.]"
                        )

                    f.write("\n\n")

                f.write("---\n\n")

        # ============================================================
        # Часть 2. max_tokens
        # ============================================================

        f.write("## Часть 2. max_tokens\n\n")

        for scenario_name, scenario_results in max_tokens_results.items():

            f.write(f"### {scenario_name}\n\n")

            for max_tokens, result in scenario_results.items():

                answer = result["answer"]
                finish_reason = result["finish_reason"]

                if answer is not None:
                    actual_tokens = count_tokens(answer)
                else:
                    actual_tokens = 0

                f.write(f"#### max_tokens = {max_tokens}\n\n")

                f.write(
                    f"- finish_reason: `{finish_reason}`\n"
                )

                f.write(
                    f"- длина финального ответа: "
                    f"≈ {actual_tokens} токенов\n\n"
                )

                f.write("**Ответ модели:**\n\n")

                if answer is not None:
                    f.write(answer.strip())
                else:
                    f.write(
                        "[Финальный ответ не сформирован: "
                        "лимит генерации был исчерпан до появления content.]"
                    )

                f.write("\n\n---\n\n")

    print(f"Результаты сохранены в {filename}")

In [ ]:
save_task3_results(temperature_results, max_tokens_results)

Результаты сохранены в task3_results.md


### Анализ результатов

Изменение `temperature` заметно повлияло на стабильность и качество ответов.

При `temperature = 0.0` и `0.5` ответы были наиболее связными и предсказуемыми. В творческом сценарии идеи отличались между запусками, но оставались логичными и структурированными. При `temperature = 1.0` разнообразие выросло, однако качество стало менее стабильным: один ответ был обрезан по лимиту, а в других появились более странные формулировки и языковые ошибки.
В логическом сценарии ответы при `temperature = 0.0` и `0.5` были подробными и последовательными. При `temperature = 1.0` общая логика сохранилась, но стало больше неточных, неестественных и ошибочных формулировок.
Эксперимент с `max_tokens` показал прямое влияние доступного бюджета генерации. При `1000` токенов модель не успевала сформировать финальный ответ, при `2000` ответ появлялся, но мог обрываться, а при `4000` ответы завершались естественно.


### Вывод

Для данной модели наиболее стабильными оказались значения `temperature = 0.0–0.5`: ответы сохраняли хорошую связность и логичность, хотя даже при низкой temperature не были полностью идентичными.

`temperature = 1.0` повышала разнообразие, но одновременно увеличивала риск менее точных и менее связных формулировок.

Параметр `max_tokens` оказался критичным для reasoning-модели: слишком маленький бюджет полностью расходуется до формирования финального ответа. В проведённом эксперименте `4000` токенов оказалось достаточным для полноценной генерации.

Таким образом, для логических и учебных задач лучше использовать низкую или среднюю `temperature` и достаточно большой `max_tokens`, а высокую `temperature` — в первую очередь для задач, где важнее разнообразие и креативность.


### Задание 4. Управление стилем и форматом ответа

Цель эксперимента — проверить, насколько точно языковая модель соблюдает требования к форме ответа, заданные непосредственно в пользовательском промпте.

Для каждого сценария используются три формата:

1. краткий ответ в 2–3 предложениях;
2. структурированный ответ в виде списка;
3. пошаговое объяснение.

Проверяются два сценария:

* объяснение технического понятия;
* практические рекомендации.

Во всех запросах используется одна и та же модель и одинаковые параметры генерации. Меняется только инструкция, задающая формат ответа.

После выполнения сравнивается соблюдение заданного формата и удобство восприятия каждого варианта.


In [ ]:
scenarios = {
    "Сценарий 1 — техническое понятие": (
        "Объясни, что такое API и зачем оно используется "
        "при разработке программных систем."
    ),

    "Сценарий 2 — практические рекомендации": (
        "Объясни, как разработчику подготовиться "
        "к техническому собеседованию."
    ),
}

In [ ]:
format_prompts = {
    "Краткий ответ": (
        "Ответь строго в 2–3 предложениях. "
        "Не используй списки, заголовки и дополнительные разделы."
    ),

    "Структурированный список": (
        "Ответь в виде маркированного списка. "
        "Используй от 4 до 6 пунктов. "
        "Не добавляй длинное вступление или заключение."
    ),

    "Пошаговое объяснение": (
        "Объясни ответ пошагово. "
        "Используй нумерованные шаги: Шаг 1, Шаг 2, Шаг 3 и так далее. "
        "Каждый шаг должен содержать краткое пояснение."
    ),
}

In [ ]:
def generate_formatted_answer(topic, format_instruction):
    """
    Генерирует ответ с заданным форматом.

    Во всех экспериментах параметры генерации одинаковы.
    Меняется только инструкция по формату.
    """

    prompt = f"""
{format_instruction}

Запрос:
{topic}
"""

    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.2,
        max_tokens=4000,
    )

    message = response.choices[0].message

    return {
        "answer": message.content,
        "finish_reason": response.choices[0].finish_reason,
    }

In [ ]:
results = {}

for scenario_name, topic in scenarios.items():

    print("=" * 100)
    print(scenario_name)
    print("=" * 100)

    results[scenario_name] = {}

    for format_name, format_instruction in format_prompts.items():

        result = generate_formatted_answer(
            topic=topic,
            format_instruction=format_instruction,
        )

        results[scenario_name][format_name] = result

        print(f"\nФормат: {format_name}")
        print(f"finish_reason: {result['finish_reason']}")
        print("-" * 80)
        print(result["answer"])

Сценарий 1 — техническое понятие

Формат: Краткий ответ
finish_reason: stop
--------------------------------------------------------------------------------
API (Application Programming Interface) — это набор правил и протоколов, позволяющих программам взаимодействовать между собой, предоставляя функции или данные без необходимости знать внутреннюю реализацию. Используя API, разработчики могут интегрировать различные сервисы, упрощать поддержку кода и обеспечивать гибкость и масштабируемость программных систем.

Формат: Структурированный список
finish_reason: stop
--------------------------------------------------------------------------------
* **Определение**: API (Application Programming Interface) — это набор стандартизированных правил и интерфейсов, позволяющих одной программе взаимодействовать с другой.
* **Взаимодействие систем**: Он используется для того, чтобы различные программные системы могли обмениваться данными и вызвать функции друг друга без необходимости знать их внутр

Реализуем несколько простых валидаторов ответов.

In [ ]:
def count_sentences(text):
    """
    Приблизительно считает количество предложений.
    """
    if not text:
        return 0

    sentences = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]

    return len(sentences)


def check_format(format_name, answer):
    """
    Выполняет простую автоматическую проверку
    соблюдения требуемого формата.
    """

    if not answer:
        return False

    if format_name == "Краткий ответ":
        sentence_count = count_sentences(answer)
        return 2 <= sentence_count <= 3

    elif format_name == "Структурированный список":
        lines = answer.splitlines()

        bullet_lines = [line for line in lines if line.strip().startswith(("-", "*", "•"))]

        return 4 <= len(bullet_lines) <= 6

    elif format_name == "Пошаговое объяснение":
        return "Шаг 1" in answer and "Шаг 2" in answer and "Шаг 3" in answer

    return False

In [ ]:
for scenario_name, scenario_results in results.items():

    print(f"\n{scenario_name}")

    for format_name, result in scenario_results.items():

        follows_format = check_format(
            format_name,
            result["answer"]
        )

        print(
            f"{format_name}: "
            f"формат соблюдён = {follows_format}"
        )


Сценарий 1 — техническое понятие
Краткий ответ: формат соблюдён = True
Структурированный список: формат соблюдён = True
Пошаговое объяснение: формат соблюдён = True

Сценарий 2 — практические рекомендации
Краткий ответ: формат соблюдён = True
Структурированный список: формат соблюдён = True
Пошаговое объяснение: формат соблюдён = True


In [ ]:
# Сохраним результат проверки внутрь results

for scenario_name, scenario_results in results.items():

    for format_name, result in scenario_results.items():

        result["format_ok"] = check_format(format_name, result["answer"])

In [ ]:
def save_task4_results(
    results,
    filename="task4_results.md"
):
    """
    Сохраняет результаты задания 4 в Markdown.
    """

    with open(filename, "w", encoding="utf-8") as f:

        f.write("# Задание 4. Результаты эксперимента\n\n")

        for scenario_name, scenario_results in results.items():

            f.write(f"## {scenario_name}\n\n")

            for format_name, result in scenario_results.items():

                answer = result["answer"]
                finish_reason = result["finish_reason"]
                format_ok = result["format_ok"]

                f.write(f"### {format_name}\n\n")

                f.write(
                    f"- finish_reason: `{finish_reason}`\n"
                )

                f.write(
                    f"- формат соблюдён: `{format_ok}`\n\n"
                )

                f.write("**Ответ модели:**\n\n")

                if answer is not None:
                    f.write(answer.strip())
                else:
                    f.write("[Финальный ответ не сформирован]")

                f.write("\n\n---\n\n")

    print(f"Результаты сохранены в {filename}")

In [ ]:
save_task4_results(results)

Результаты сохранены в task4_results.md


### Анализ результатов

Во всех шести запросах модель корректно соблюдала заданный формат ответа: краткие ответы укладывались в 2–3 предложения, структурированные ответы были оформлены списком, а пошаговые — через последовательные шаги.
По удобству восприятия форматы различаются: краткий ответ лучше подходит для быстрого ознакомления, список — для компактного обзора нескольких пунктов, а пошаговое объяснение — для более подробного обучения и практических инструкций.


### Вывод

Эксперимент показал, что формат ответа можно достаточно надёжно контролировать через явные инструкции в промпте.

Наиболее подходящий формат зависит от задачи: краткий ответ удобен для быстрых справок, список — для структурированного представления информации, а пошаговый формат — для объяснения процессов и последовательности действий.


### Задание 5. Комбинированный сценарий

Цель эксперимента — проверить устойчивость поведения модели, когда одновременно используются несколько способов управления генерацией:

* system prompt с заданной ролью;
* многошаговый диалог и информация из предыдущих сообщений;
* ограничения на формат ответа;
* изменение `temperature`.

Используются два сценария:

1. объяснение переобучения модели машинного обучения;
2. объяснение синхронного и асинхронного выполнения в Python.

Каждый сценарий состоит из двух шагов: сначала модель объясняет тему, затем пользователь задаёт уточняющий вопрос, который требует учитывать предыдущий контекст и соблюдать строгий формат ответа.

Каждый диалог выполняется при `temperature = 0.2` и `temperature = 1.0`. Это позволяет сравнить устойчивость роли, контекста и формата при более детерминированной и более случайной генерации.

In [ ]:
SYSTEM_PROMPT = (
    "Ты преподаватель программирования и машинного обучения. "
    "Объясняй материал понятно, последовательно и простым языком. "
    "Используй технические термины только там, где они действительно нужны. "
    "Учитывай информацию из предыдущих сообщений диалога и строго соблюдай "
    "требования пользователя к формату ответа."
)

In [ ]:
scenarios = {
    "Сценарий 1 — переобучение": {
        "first_prompt": (
            "Объясни, что такое переобучение модели машинного обучения. "
            "Для примера используй задачу классификации фотографий кошек и собак."
        ),

        "followup_prompt": (
            "Теперь, опираясь на предыдущее объяснение и используя тот же пример "
            "с кошками и собаками, назови способы уменьшить переобучение. "
            "Ответь строго в виде 3 маркированных пунктов. "
            "Каждый пункт должен состоять из одного предложения."
        ),
    },

    "Сценарий 2 — асинхронность": {
        "first_prompt": (
            "Объясни разницу между синхронным и асинхронным выполнением в Python. "
            "Используй пример приложения StudyBot, которое должно отправить "
            "5 независимых запросов к внешнему API."
        ),

        "followup_prompt": (
            "Теперь, опираясь на предыдущее объяснение, расскажи, как перевести "
            "StudyBot на асинхронную работу. Ответь строго тремя шагами "
            "с названиями 'Шаг 1', 'Шаг 2' и 'Шаг 3'. "
            "Каждый шаг должен состоять из одного предложения."
        ),
    },
}

Функция для одного многошагового диалога:

In [ ]:
def run_conversation(first_prompt, followup_prompt, temperature):
    """
    Выполняет двухшаговый диалог с моделью.

    Второй запрос получает всю историю разговора:
    system prompt -> первый user prompt -> первый ответ ->
    уточняющий user prompt.
    """

    # Начинаем диалог с system prompt и первого сообщения пользователя
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": first_prompt,
        },
    ]

    # Первый ответ модели
    first_response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=4000,
    )

    first_answer = first_response.choices[0].message.content

    # Добавляем первый ответ в историю диалога
    messages.append({
        "role": "assistant",
        "content": first_answer,
    })

    # Добавляем уточняющий вопрос пользователя
    messages.append({
        "role": "user",
        "content": followup_prompt,
    })

    # Второй ответ уже получает весь предыдущий контекст
    second_response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=4000,
    )

    second_answer = second_response.choices[0].message.content

    return {
        "first_answer": first_answer,
        "first_finish_reason": first_response.choices[0].finish_reason,
        "second_answer": second_answer,
        "second_finish_reason": second_response.choices[0].finish_reason,
    }

In [ ]:
temperatures = [0.2, 1.0]

results = {}

for scenario_name, scenario in scenarios.items():

    print("=" * 100)
    print(scenario_name)
    print("=" * 100)

    results[scenario_name] = {}

    for temperature in temperatures:

        print(f"\nTEMPERATURE = {temperature}")
        print("-" * 100)

        result = run_conversation(
            first_prompt=scenario["first_prompt"],
            followup_prompt=scenario["followup_prompt"],
            temperature=temperature,
        )

        results[scenario_name][temperature] = result

        print("\nШАГ 1:")
        print(result["first_answer"])

        print("\nШАГ 2:")
        print(result["second_answer"])

        print(
            "\nfinish_reason:",
            result["first_finish_reason"],
            "/",
            result["second_finish_reason"],
        )

Сценарий 1 — переобучение

TEMPERATURE = 0.2
----------------------------------------------------------------------------------------------------

ШАГ 1:
# Переобучение в машинном обучении

## Что это такое?

**Переобучение (overfitting)** — это ситуация, когда модель машинного обучения слишком хорошо подстраивается под данные из обучающей выборки, но плохо работает на новых, неизвестных данных.

Представь, что ты учишь ребенка распознавать кошек и собак по фотографиям. Если ты даешь ему слишком много похожих картинок и он запоминает каждую деталь — уникальное пятнышко, цвет шерсти, положение ушей — то он будет отлично работать на этих конкретных картинках, но если ты покажешь ему фотографию кошки, которую никогда не видел раньше, он может ошибиться.

## Пример с кошками и собаками

Допустим, мы создаём модель для классификации фото:

- **Обучающая выборка**: 1000 фото кошек и 1000 фото собак
- **Сложная модель**: очень глубокая нейросеть с тысячами параметров

Модель начинает учить не

Напишем простую автоматическую проверку, которая проверяет:

- сохранился ли контекст;
- соблюдён ли формат второго ответа.

Окончательная оценка устойчивости роли, контекста и формата проводится по содержанию фактически полученных ответов.

In [ ]:
def check_task5_result(scenario_name, answer):
    """
    Выполняет простую проверку сохранения контекста
    и соблюдения формата второго ответа.
    """

    if not answer:
        return {"context_ok": False, "format_ok": False}

    answer_lower = answer.lower()

    # -------------------------------------------------------
    # Сценарий 1
    # -------------------------------------------------------

    if scenario_name == "Сценарий 1 — переобучение":

        # Проверяем, что модель продолжает использовать
        # пример с кошками/собаками.
        context_ok = "кош" in answer_lower or "собак" in answer_lower

        # Считаем маркированные пункты.
        bullet_lines = [line for line in answer.splitlines() if line.strip().startswith(("-", "*", "•"))]

        format_ok = len(bullet_lines) == 3

    # -------------------------------------------------------
    # Сценарий 2
    # -------------------------------------------------------

    elif scenario_name == "Сценарий 2 — асинхронность":

        # Проверяем использование StudyBot
        # или информации про несколько API-запросов.
        context_ok = "studybot" in answer_lower or "api" in answer_lower

        format_ok = (
            "Шаг 1" in answer
            and "Шаг 2" in answer
            and "Шаг 3" in answer
            and "Шаг 4" not in answer
        )

    else:
        context_ok = False
        format_ok = False

    return {"context_ok": context_ok, "format_ok": format_ok}

In [ ]:
for scenario_name, scenario_results in results.items():

    for temperature, result in scenario_results.items():

        checks = check_task5_result(scenario_name, result["second_answer"])

        result["context_ok"] = checks["context_ok"]
        result["format_ok"] = checks["format_ok"]

In [ ]:
for scenario_name, scenario_results in results.items():

    print(f"\n{scenario_name}")

    for temperature, result in scenario_results.items():

        print(
            f"temperature={temperature}: "
            f"context={result['context_ok']}, "
            f"format={result['format_ok']}"
        )


Сценарий 1 — переобучение
temperature=0.2: context=False, format=True
temperature=1.0: context=True, format=True

Сценарий 2 — асинхронность
temperature=0.2: context=False, format=True
temperature=1.0: context=False, format=True


In [ ]:
def save_task5_results(
    results,
    filename="task5_results.md"
):
    """
    Сохраняет результаты комбинированного эксперимента
    в Markdown-файл.
    """

    with open(filename, "w", encoding="utf-8") as f:

        f.write("# Задание 5. Результаты эксперимента\n\n")

        for scenario_name, scenario_results in results.items():

            f.write(f"## {scenario_name}\n\n")

            for temperature, result in scenario_results.items():

                f.write(
                    f"### temperature = {temperature}\n\n"
                )

                # -------------------------------
                # Первый шаг
                # -------------------------------

                f.write("#### Шаг 1 — первоначальный ответ\n\n")

                f.write(
                    f"- finish_reason: "
                    f"`{result['first_finish_reason']}`\n\n"
                )

                if result["first_answer"] is not None:
                    f.write(result["first_answer"].strip())
                else:
                    f.write(
                        "[Финальный ответ не сформирован]"
                    )

                f.write("\n\n")

                # -------------------------------
                # Второй шаг
                # -------------------------------

                f.write("#### Шаг 2 — уточняющий вопрос\n\n")

                f.write(
                    f"- finish_reason: "
                    f"`{result['second_finish_reason']}`\n"
                )

                f.write(
                    f"- контекст сохранён: "
                    f"`{result['context_ok']}`\n"
                )

                f.write(
                    f"- формат соблюдён: "
                    f"`{result['format_ok']}`\n\n"
                )

                if result["second_answer"] is not None:
                    f.write(result["second_answer"].strip())
                else:
                    f.write(
                        "[Финальный ответ не сформирован]"
                    )

                f.write("\n\n---\n\n")

    print(f"Результаты сохранены в {filename}")

In [ ]:
save_task5_results(results)

Результаты сохранены в task5_results.md


### Анализ результатов

Комбинированный сценарий в целом оказался устойчивым: модель сохраняла роль преподавателя и во всех случаях соблюдала требуемый формат второго ответа.
С контекстом результаты были менее стабильными. В сценарии с переобучением при `temperature = 0.2` модель дала корректные способы борьбы с переобучением, но не продолжила явно использовать пример с кошками и собаками, поэтому автоматическая проверка контекста дала `False`. При `temperature = 1.0` нужный пример сохранился.
В сценарии с асинхронностью формат соблюдался при обеих температурах, но модель не повторяла название `StudyBot`, из-за чего автоматическая проверка контекста также дала `False`, хотя она явно опиралась на предыдущую ситуацию с несколькими API-запросами.
При `temperature = 1.0` заметно ухудшилось качество формулировок: появились странные слова, грамматические ошибки и менее точные технические выражения.


### Вывод

Из трёх контролируемых аспектов наиболее устойчивым оказался **формат ответа** — модель соблюдала его во всех случаях.

**Стиль преподавателя** в целом также сохранялся, но при высокой `temperature` качество языка и точность формулировок ухудшались.

Наименее устойчивым оказался **явный перенос деталей контекста**: модель могла учитывать смысл предыдущего сообщения, но не повторять конкретные примеры или названия. Поэтому в данном эксперименте первым «ломался» именно контекст, а высокая `temperature` дополнительно снижала качество формулировок.


### Общий вывод

Эксперименты показали, что поведение языковой модели заметно зависит от способа постановки задачи и параметров генерации. System prompt управляет стилем и уровнем детализации, явные инструкции хорошо контролируют формат ответа, а `temperature` влияет на стабильность и вариативность генерации. Увеличение контекста до 500 токенов не привело к потере информации, а слишком маленький `max_tokens` для reasoning-модели оказался критичным и мог полностью прерывать формирование финального ответа.

В целом наиболее устойчивые результаты получались при чётких инструкциях, низкой или средней `temperature` и достаточном лимите генерации.
